#**Connect drive**

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/thesis_DR")

files = [
    "aptos_retfound_mlp_heads.pt",
    "aptos_retfound_temperature_scaling.json",
    "aptos_retfound_ltt_risk_control.csv"
]

for f in files:
    path = PROJECT_DIR / f
    print(f, "✅ FOUND" if path.exists() else "❌ MISSING")

aptos_retfound_mlp_heads.pt ✅ FOUND
aptos_retfound_temperature_scaling.json ✅ FOUND
aptos_retfound_ltt_risk_control.csv ✅ FOUND


#**Load Dataset Messidor-2**

In [3]:
# ============================================================
# DOWNLOAD MESSIDOR-2 TO TEMPORARY COLAB STORAGE
# ============================================================

!pip -q install gdown

from pathlib import Path
import zipfile
import gdown

ZIP_PATH = Path("/content/MESSIDOR2.zip")
DATA_DIR = Path("/content/MESSIDOR2")

# RETFound official benchmark MESSIDOR2 archive
FILE_ID = "1vOLBUK9xdzNV8eVkRjVdNrRwhPfaOmda"

# Download
if not ZIP_PATH.exists():
    gdown.download(
        id=FILE_ID,
        output=str(ZIP_PATH),
        quiet=False
    )

# Extract
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not any(DATA_DIR.iterdir()):
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(DATA_DIR)

print("\n✅ Messidor-2 downloaded and extracted")
print("Dataset location:", DATA_DIR)

# Quick inspection only
all_files = [p for p in DATA_DIR.rglob("*") if p.is_file()]

image_files = [
    p for p in all_files
    if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".tif", ".tiff"]
]

print("Total files:", len(all_files))
print("Image files:", len(image_files))

print("\nTop-level contents:")
for p in DATA_DIR.iterdir():
    print(" -", p.name)

Downloading...
From (original): https://drive.google.com/uc?id=1vOLBUK9xdzNV8eVkRjVdNrRwhPfaOmda
From (redirected): https://drive.google.com/uc?id=1vOLBUK9xdzNV8eVkRjVdNrRwhPfaOmda&confirm=t&uuid=a46eaa57-0d41-4617-bcc8-20a3d9c4b4d5
To: /content/MESSIDOR2.zip
100%|██████████| 2.51G/2.51G [00:42<00:00, 59.5MB/s]



✅ Messidor-2 downloaded and extracted
Dataset location: /content/MESSIDOR2
Total files: 1745
Image files: 1744

Top-level contents:
 - MESSIDOR2


In [4]:
# ============================================================
# MESSIDOR-2 RAW STRUCTURE CHECK
# ============================================================

from pathlib import Path
from collections import Counter

ROOT = Path("/content/MESSIDOR2/MESSIDOR2")

all_files = [p for p in ROOT.rglob("*") if p.is_file()]

image_exts = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}

image_files = [
    p for p in all_files
    if p.suffix.lower() in image_exts
]

non_image_files = [
    p for p in all_files
    if p.suffix.lower() not in image_exts
]

print("========== MESSIDOR-2 STRUCTURE ==========\n")

print("Root:", ROOT)
print("Total files:", len(all_files))
print("Image files:", len(image_files))
print("Non-image files:", len(non_image_files))

print("\nFile extensions:")
for ext, count in Counter(
    p.suffix.lower() for p in all_files
).most_common():
    print(ext if ext else "[no extension]", ":", count)

print("\nNon-image files:")
for p in non_image_files:
    print(" -", p.relative_to(ROOT))

print("\nFirst 20 image filenames:")
for p in image_files[:20]:
    print(" -", p.relative_to(ROOT))

print("\nFolders:")
folders = sorted({
    str(p.parent.relative_to(ROOT))
    for p in all_files
})

for f in folders[:50]:
    print(" -", f)

========== MESSIDOR-2 STRUCTURE ==========

Root: /content/MESSIDOR2/MESSIDOR2
Total files: 1745
Image files: 1744
Non-image files: 1

File extensions:
.png : 1744
.txt : 1

Non-image files:
 - readme.txt

First 20 image filenames:
 - val/eproliferativedr/20051214_40912_0100_PP.png
 - val/eproliferativedr/IM004346.png
 - val/eproliferativedr/IM001941.png
 - val/eproliferativedr/20051208_42314_0400_PP.png
 - val/eproliferativedr/20051021_52127_0100_PP.png
 - val/dseveredr/20051216_45226_0200_PP.png
 - val/dseveredr/20051212_41144_0400_PP.png
 - val/dseveredr/IM003481.png
 - val/dseveredr/20051020_45068_0100_PP.png
 - val/dseveredr/20060523_48199_0100_PP.png
 - val/dseveredr/20051205_35276_0400_PP.png
 - val/dseveredr/20051214_52242_0100_PP.png
 - val/dseveredr/20060412_62936_0200_PP.png
 - val/dseveredr/20060530_55724_0100_PP.png
 - val/dseveredr/20051214_57154_0100_PP.png
 - val/dseveredr/20051020_62385_0100_PP.png
 - val/bmilddr/IM001681.png
 - val/bmilddr/IM000406.png
 - val/bmilddr/

In [5]:
# ============================================================
# MESSIDOR-2 README + CLASS DISTRIBUTION
# ============================================================

from pathlib import Path
from collections import Counter

ROOT = Path("/content/MESSIDOR2/MESSIDOR2")

# ------------------------------------------------------------
# 1. README
# ------------------------------------------------------------

readme_path = ROOT / "readme.txt"

print("========== README ==========\n")

with open(readme_path, "r", errors="ignore") as f:
    print(f.read())


# ------------------------------------------------------------
# 2. COUNT BY SPLIT AND CLASS
# ------------------------------------------------------------

image_exts = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}

rows = []

for split in ["train", "val", "test"]:

    split_dir = ROOT / split

    for class_dir in sorted(split_dir.iterdir()):

        if not class_dir.is_dir():
            continue

        images = [
            p for p in class_dir.iterdir()
            if p.suffix.lower() in image_exts
        ]

        rows.append({
            "split": split,
            "class": class_dir.name,
            "count": len(images)
        })


import pandas as pd

counts = pd.DataFrame(rows)

print("\n========== SPLIT / CLASS COUNTS ==========\n")
display(counts.pivot(
    index="class",
    columns="split",
    values="count"
).fillna(0).astype(int))


print("\n========== TOTAL BY SPLIT ==========\n")
display(
    counts.groupby("split")["count"]
    .sum()
    .reset_index()
)


print("\n========== TOTAL BY CLASS ==========\n")
display(
    counts.groupby("class")["count"]
    .sum()
    .reset_index()
)

print("\nTOTAL IMAGES:", counts["count"].sum())

========== README ==========

This folder includes the preprocessed images from Messidor-2 (https://www.adcis.net/en/third-party/messidor2/)

Please refer to the Messidor-2 website for more information about the dataset.


========== SPLIT / CLASS COUNTS ==========



split,test,train,val
class,,,
anodr,306,568,143
bmilddr,81,151,38
cmoderatedr,105,193,49
dseveredr,23,41,11
eproliferativedr,11,19,5



========== TOTAL BY SPLIT ==========



,split,count
0,test,526
1,train,972
2,val,246



========== TOTAL BY CLASS ==========



,class,count
0,anodr,1017
1,bmilddr,270
2,cmoderatedr,347
3,dseveredr,75
4,eproliferativedr,35



TOTAL IMAGES: 1744


#**Audit Dataset**

In [6]:
# ============================================================
# MESSIDOR-2 IMAGE + EXACT DUPLICATE AUDIT
# DISPLAY ONLY - NOTHING SAVED
# ============================================================

from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import pandas as pd
import hashlib

ROOT = Path("/content/MESSIDOR2/MESSIDOR2")

CLASS_TO_GRADE = {
    "anodr": 0,
    "bmilddr": 1,
    "cmoderatedr": 2,
    "dseveredr": 3,
    "eproliferativedr": 4
}

image_exts = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}

image_files = sorted([
    p for p in ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in image_exts
])

print("Total images found:", len(image_files))


# ============================================================
# AUDIT
# ============================================================

rows = []

for path in tqdm(image_files):

    split = path.parent.parent.name
    class_name = path.parent.name
    grade = CLASS_TO_GRADE[class_name]

    readable = True
    width = None
    height = None
    mode = None
    error = ""

    try:
        with Image.open(path) as img:
            img.verify()

        with Image.open(path) as img:
            width, height = img.size
            mode = img.mode

    except Exception as e:
        readable = False
        error = str(e)

    # SHA-256 exact file hash
    sha256 = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            sha256.update(chunk)

    rows.append({
        "filename": path.name,
        "path": str(path),
        "split": split,
        "class_name": class_name,
        "grade": grade,
        "binary_label": int(grade >= 2),
        "readable": readable,
        "width": width,
        "height": height,
        "mode": mode,
        "sha256": sha256.hexdigest(),
        "error": error
    })

audit = pd.DataFrame(rows)


# ============================================================
# BASIC RESULTS
# ============================================================

print("\n========== IMAGE INTEGRITY ==========")

print("Total images:", len(audit))
print("Readable:", int(audit["readable"].sum()))
print("Unreadable:", int((~audit["readable"]).sum()))


print("\n========== IMAGE DIMENSIONS ==========")

display(
    audit.groupby(
        ["width", "height", "mode"],
        dropna=False
    )
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)


# ============================================================
# EXACT DUPLICATES
# ============================================================

hash_counts = audit["sha256"].value_counts()

duplicate_hashes = hash_counts[
    hash_counts > 1
].index

dup = audit[
    audit["sha256"].isin(duplicate_hashes)
].copy()

print("\n========== EXACT DUPLICATE AUDIT ==========")

print("Unique SHA-256 images:", audit["sha256"].nunique())
print("Exact duplicate groups:", len(duplicate_hashes))
print("Images involved in duplicate groups:", len(dup))


# ============================================================
# LABEL CONFLICTS
# ============================================================

same_grade_groups = 0
grade_conflict_groups = 0
binary_conflict_groups = 0

for sha, group in dup.groupby("sha256"):

    if group["grade"].nunique() == 1:
        same_grade_groups += 1
    else:
        grade_conflict_groups += 1

    if group["binary_label"].nunique() > 1:
        binary_conflict_groups += 1


print("Same-grade duplicate groups:", same_grade_groups)
print("Original-grade conflict groups:", grade_conflict_groups)
print("Binary-label conflict groups:", binary_conflict_groups)


# ============================================================
# DUPLICATE GROUP SIZES
# ============================================================

if len(duplicate_hashes) > 0:

    print("\nDuplicate group sizes:")

    display(
        hash_counts[
            hash_counts > 1
        ]
        .value_counts()
        .sort_index()
        .rename_axis("group_size")
        .reset_index(name="number_of_groups")
    )


# ============================================================
# BINARY DISTRIBUTION
# ============================================================

print("\n========== BINARY LABEL DISTRIBUTION ==========")

binary_counts = (
    audit["binary_label"]
    .value_counts()
    .sort_index()
)

print(
    "Non-referable (0):",
    int(binary_counts.get(0, 0))
)

print(
    "Referable (1):",
    int(binary_counts.get(1, 0))
)


print("\nDISPLAY ONLY - NO FILE SAVED.")

Total images found: 1744


  0%|          | 0/1744 [00:00<?, ?it/s]


========== IMAGE INTEGRITY ==========
Total images: 1744
Readable: 1744
Unreadable: 0

========== IMAGE DIMENSIONS ==========


,width,height,mode,count
10,1452,1452,RGB,571
0,904,904,RGB,311
5,1384,1384,RGB,269
4,1380,1380,RGB,201
6,1388,1388,RGB,129
1,908,908,RGB,113
2,912,912,RGB,103
11,1456,1456,RGB,29
3,1376,1376,RGB,8
7,1392,1392,RGB,6



========== EXACT DUPLICATE AUDIT ==========
Unique SHA-256 images: 1740
Exact duplicate groups: 4
Images involved in duplicate groups: 8
Same-grade duplicate groups: 4
Original-grade conflict groups: 0
Binary-label conflict groups: 0

Duplicate group sizes:


,group_size,number_of_groups
0,2,4



========== BINARY LABEL DISTRIBUTION ==========
Non-referable (0): 1287
Referable (1): 457

DISPLAY ONLY - NO FILE SAVED.


In [7]:
# ============================================================
# FINAL MESSIDOR-2 EXTERNAL COHORT
# Remove only repeated exact copies
# ============================================================

messidor = audit.copy()

# Deterministic ordering
messidor = messidor.sort_values(
    ["sha256", "path"]
).reset_index(drop=True)

# Keep first occurrence of each exact image
messidor["status"] = "KEEP"

duplicate_extra = messidor.duplicated(
    subset="sha256",
    keep="first"
)

messidor.loc[
    duplicate_extra,
    "status"
] = "DROP_EXACT_DUPLICATE"


# ============================================================
# FINAL COHORT
# ============================================================

final_messidor = messidor[
    messidor["status"] == "KEEP"
].copy()

print("========== FINAL MESSIDOR-2 COHORT ==========\n")

print("Original images:", len(messidor))
print(
    "Exact duplicate copies excluded:",
    (messidor["status"] == "DROP_EXACT_DUPLICATE").sum()
)
print("Final unique images:", len(final_messidor))


print("\n========== GRADE DISTRIBUTION ==========\n")

grade_counts = (
    final_messidor["grade"]
    .value_counts()
    .sort_index()
)

for grade in range(5):
    print(
        f"Grade {grade}:",
        int(grade_counts.get(grade, 0))
    )


print("\n========== BINARY DISTRIBUTION ==========\n")

binary_counts = (
    final_messidor["binary_label"]
    .value_counts()
    .sort_index()
)

print(
    "Non-referable:",
    int(binary_counts.get(0, 0))
)

print(
    "Referable:",
    int(binary_counts.get(1, 0))
)

print("\nFinal cohort ready for external validation.")

========== FINAL MESSIDOR-2 COHORT ==========

Original images: 1744
Exact duplicate copies excluded: 4
Final unique images: 1740

========== GRADE DISTRIBUTION ==========

Grade 0: 1017
Grade 1: 268
Grade 2: 345
Grade 3: 75
Grade 4: 35

========== BINARY DISTRIBUTION ==========

Non-referable: 1285
Referable: 455

Final cohort ready for external validation.


#**Messidor-2 Embedding Extraction**

In [8]:
# ============================================================
# MESSIDOR-2 EMBEDDING EXTRACTION
# RETFound-Green checkpoint saved permanently in Drive
# ============================================================

!pip -q install timm==0.9.12

import numpy as np
import torch
import timm
import hashlib

from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm


# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/thesis_DR")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

ROOT = Path("/content/MESSIDOR2/MESSIDOR2")

# PERMANENT checkpoint in Drive
CHECKPOINT = PROJECT_DIR / "retfoundgreen_statedict.pth"

# Final embeddings saved in Drive
OUTPUT_PATH = PROJECT_DIR / "messidor2_retfound_green_embeddings.npz"


# ============================================================
# DOWNLOAD RETFOUND-GREEN ONLY ONCE
# ============================================================

if not CHECKPOINT.exists():

    print("RETFound-Green checkpoint not in Drive.")
    print("Downloading once...")

    !wget -q \
    https://github.com/justinengelmann/RETFound_Green/releases/download/v0.1/retfoundgreen_statedict.pth \
    -O "/content/drive/MyDrive/thesis_DR/retfoundgreen_statedict.pth"

    print("✅ Saved permanently to Drive")

else:
    print("✅ RETFound-Green already exists in Drive")


print("Checkpoint:", CHECKPOINT)


# ============================================================
# GPU CHECK
# ============================================================

if not torch.cuda.is_available():

    raise RuntimeError(
        "GPU is not enabled. In Colab select Runtime > Change runtime type > T4 GPU."
    )

device = torch.device("cuda")

print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# RECREATE FINAL MESSIDOR COHORT
# 1744 images -> remove repeated exact copies
# ============================================================

CLASS_TO_GRADE = {
    "anodr": 0,
    "bmilddr": 1,
    "cmoderatedr": 2,
    "dseveredr": 3,
    "eproliferativedr": 4
}

IMAGE_EXTS = {
    ".png", ".jpg", ".jpeg", ".tif", ".tiff"
}

image_files = sorted([
    p for p in ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS
])


records = []

for path in tqdm(
    image_files,
    desc="Checking exact duplicates"
):

    sha = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            sha.update(chunk)

    class_name = path.parent.name
    grade = CLASS_TO_GRADE[class_name]

    records.append({
        "path": str(path),
        "sha256": sha.hexdigest(),
        "grade": grade,
        "binary_label": int(grade >= 2)
    })


import pandas as pd

messidor = pd.DataFrame(records)

messidor = messidor.sort_values(
    ["sha256", "path"]
).reset_index(drop=True)

# Keep one copy of each exact image
final_messidor = (
    messidor
    .drop_duplicates(
        subset="sha256",
        keep="first"
    )
    .reset_index(drop=True)
)

print("\nOriginal images:", len(messidor))
print(
    "Exact duplicate copies excluded:",
    len(messidor) - len(final_messidor)
)
print(
    "Final external cohort:",
    len(final_messidor)
)

assert len(final_messidor) == 1740


# ============================================================
# SAME LOCKED PREPROCESSING AS APTOS
# ============================================================

def preprocess_retina(path):

    img = Image.open(path).convert("RGB")

    arr = np.asarray(img)

    # Remove obvious exterior black border
    mask = np.max(arr, axis=2) > 10

    if mask.any():

        ys, xs = np.where(mask)

        left = xs.min()
        right = xs.max() + 1
        top = ys.min()
        bottom = ys.max() + 1

        img = img.crop(
            (left, top, right, bottom)
        )

    # Preserve aspect ratio
    w, h = img.size
    side = max(w, h)

    # Black square padding
    square = Image.new(
        "RGB",
        (side, side),
        (0, 0, 0)
    )

    square.paste(
        img,
        (
            (side - w) // 2,
            (side - h) // 2
        )
    )

    # Same RETFound resolution
    square = square.resize(
        (392, 392),
        Image.Resampling.LANCZOS
    )

    return square


transform = transforms.Compose([

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])


# ============================================================
# DATASET
# ============================================================

class MessidorDataset(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        img = preprocess_retina(
            row["path"]
        )

        img = transform(img)

        return img


dataset = MessidorDataset(
    final_messidor
)

loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(
    "\nImages for embedding:",
    len(dataset)
)


# ============================================================
# LOAD FROZEN RETFOUND-GREEN
# ============================================================

model = timm.create_model(
    "vit_small_patch14_reg4_dinov2",
    img_size=(392, 392),
    num_classes=0,
    checkpoint_path=str(CHECKPOINT)
)

model.global_pool = "avg"

model = model.to(device)
model.eval()

for param in model.parameters():
    param.requires_grad = False


trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    "Trainable parameters:",
    trainable
)

assert trainable == 0


# ============================================================
# EXTRACT EMBEDDINGS
# ============================================================

all_embeddings = []

with torch.inference_mode():

    for images in tqdm(
        loader,
        desc="Extracting embeddings"
    ):

        images = images.to(
            device,
            non_blocking=True
        )

        features = model(images)

        all_embeddings.append(
            features.cpu().numpy()
        )


embeddings = np.concatenate(
    all_embeddings,
    axis=0
).astype(np.float32)


# ============================================================
# FINAL CHECK
# ============================================================

print("\n========== RESULT ==========\n")

print(
    "Embedding shape:",
    embeddings.shape
)

print(
    "NaN values:",
    np.isnan(embeddings).sum()
)

print(
    "Inf values:",
    np.isinf(embeddings).sum()
)


assert embeddings.shape == (1740, 384)
assert np.isnan(embeddings).sum() == 0
assert np.isinf(embeddings).sum() == 0


# ============================================================
# SAVE EMBEDDINGS
# ============================================================

image_ids = np.array([
    str(
        Path(path).relative_to(ROOT)
    )
    for path in final_messidor["path"]
])

grades = final_messidor[
    "grade"
].to_numpy(dtype=np.int64)

binary_labels = final_messidor[
    "binary_label"
].to_numpy(dtype=np.int64)


np.savez_compressed(
    OUTPUT_PATH,
    embeddings=embeddings,
    image_id=image_ids,
    original_grade=grades,
    binary_label=binary_labels
)


print("\n✅ Saved:")
print(OUTPUT_PATH)

print("\nRETFound-Green checkpoint remains permanently at:")
print(CHECKPOINT)

print("\n========== COMPLETE ==========")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 47.1 MB/s eta 0:00:00
RETFound-Green checkpoint not in Drive.
✅ Saved permanently to Drive
Checkpoint: /content/drive/MyDrive/thesis_DR/retfoundgreen_statedict.pth
GPU: Tesla T4


Checking exact duplicates:   0%|          | 0/1744 [00:00<?, ?it/s]


Original images: 1744
Exact duplicate copies excluded: 4
Final external cohort: 1740

Images for embedding: 1740
Trainable parameters: 0


Extracting embeddings:   0%|          | 0/109 [00:00<?, ?it/s]


========== RESULT ==========

Embedding shape: (1740, 384)
NaN values: 0
Inf values: 0

✅ Saved:
/content/drive/MyDrive/thesis_DR/messidor2_retfound_green_embeddings.npz

RETFound-Green checkpoint remains permanently at:
/content/drive/MyDrive/thesis_DR/retfoundgreen_statedict.pth

========== COMPLETE ==========


**APTOS MLP Head Check**

In [9]:
import torch

HEAD_PATH = "/content/drive/MyDrive/thesis_DR/aptos_retfound_mlp_heads.pt"

heads = torch.load(
    HEAD_PATH,
    map_location="cpu",
    weights_only=False
)

print("Object type:", type(heads))

if isinstance(heads, dict):
    print("Top-level keys:", heads.keys())

    for key, value in heads.items():
        print("\nKEY:", key)
        print("TYPE:", type(value))

        if isinstance(value, dict):
            print("SUBKEYS:", value.keys())

Object type: <class 'dict'>
Top-level keys: dict_keys(['backbone', 'encoder_status', 'embedding_dim', 'head_architecture', 'training_protocol', 'hpo_configs', 'seeds', 'heads'])

KEY: backbone
TYPE: <class 'str'>

KEY: encoder_status
TYPE: <class 'str'>

KEY: embedding_dim
TYPE: <class 'int'>

KEY: head_architecture
TYPE: <class 'dict'>
SUBKEYS: dict_keys(['input_dim', 'hidden_dim', 'activation', 'dropout', 'output_dim'])

KEY: training_protocol
TYPE: <class 'dict'>
SUBKEYS: dict_keys(['loss', 'optimizer', 'batch_size', 'max_epochs', 'early_stopping_patience', 'inner_cv_folds', 'inner_cv_stratification', 'hpo_selection_metric', 'final_epoch_rule', 'class_weighting'])

KEY: hpo_configs
TYPE: <class 'list'>

KEY: seeds
TYPE: <class 'list'>

KEY: heads
TYPE: <class 'dict'>
SUBKEYS: dict_keys([42, 43, 44, 45, 46])


In [10]:
import torch

HEAD_PATH = "/content/drive/MyDrive/thesis_DR/aptos_retfound_mlp_heads.pt"

data = torch.load(
    HEAD_PATH,
    map_location="cpu",
    weights_only=False
)

for seed in [42, 43, 44, 45, 46]:
    print(f"\n========== SEED {seed} ==========")

    obj = data["heads"][seed]

    print("Type:", type(obj))

    if isinstance(obj, dict):
        print("Keys:", obj.keys())

        for k, v in obj.items():
            print(
                k,
                "->",
                type(v),
                getattr(v, "shape", "")
            )

    break


========== SEED 42 ==========
Type: <class 'dict'>
Keys: dict_keys(['state_dict', 'selected_config', 'selected_learning_rate', 'selected_weight_decay', 'selected_epochs', 'development_images', 'development_non_referable', 'development_referable', 'final_training_loss', 'final_loss_history', 'hpo_results'])
state_dict -> <class 'dict'> 
selected_config -> <class 'str'> 
selected_learning_rate -> <class 'float'> 
selected_weight_decay -> <class 'float'> 
selected_epochs -> <class 'int'> 
development_images -> <class 'int'> 
development_non_referable -> <class 'int'> 
development_referable -> <class 'int'> 
final_training_loss -> <class 'float'> 
final_loss_history -> <class 'list'> 
hpo_results -> <class 'list'> 


#**Messidor-2 Prediction**

In [11]:
# ============================================================
# MESSIDOR-2 PREDICTION
# Apply 5 frozen APTOS MLP heads
# ============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/thesis_DR")

EMBED_PATH = PROJECT_DIR / "messidor2_retfound_green_embeddings.npz"
HEAD_PATH = PROJECT_DIR / "aptos_retfound_mlp_heads.pt"

OUTPUT_PATH = PROJECT_DIR / "messidor2_retfound_raw_predictions.csv"

SEEDS = [42, 43, 44, 45, 46]


# ============================================================
# LOAD EMBEDDINGS
# ============================================================

data = np.load(
    EMBED_PATH,
    allow_pickle=True
)

X = torch.tensor(
    data["embeddings"],
    dtype=torch.float32
)

image_ids = data["image_id"]
grades = data["original_grade"].astype(int)
labels = data["binary_label"].astype(int)

print("Images:", len(X))
print("Embedding shape:", tuple(X.shape))


# ============================================================
# LOAD FROZEN APTOS HEADS
# ============================================================

saved = torch.load(
    HEAD_PATH,
    map_location="cpu",
    weights_only=False
)

print("Heads:", saved["heads"].keys())


# ============================================================
# PREDICT
# ============================================================

rows = []

for seed in SEEDS:

    state = saved["heads"][seed]["state_dict"]

    # --------------------------------------------------------
    # Find the two Linear layers using their tensor shapes.
    # Architecture is locked:
    # 384 -> 512 -> ReLU -> Dropout(0.1) -> 1
    # --------------------------------------------------------

    w1 = None
    b1 = None
    w2 = None
    b2 = None

    for name, tensor in state.items():

        shape = tuple(tensor.shape)

        if shape == (512, 384):
            w1 = tensor

        elif shape == (512,):
            b1 = tensor

        elif shape == (1, 512):
            w2 = tensor

        elif shape == (1,):
            b2 = tensor


    assert w1 is not None
    assert b1 is not None
    assert w2 is not None
    assert b2 is not None


    # --------------------------------------------------------
    # FROZEN FORWARD PASS
    # Dropout is NOT active during inference
    # --------------------------------------------------------

    with torch.no_grad():

        hidden = F.linear(
            X,
            w1,
            b1
        )

        hidden = F.relu(hidden)

        logits = F.linear(
            hidden,
            w2,
            b2
        ).squeeze(1)

        probabilities = torch.sigmoid(
            logits
        )


    logits = logits.numpy()
    probabilities = probabilities.numpy()


    print(
        f"Seed {seed}:",
        f"logit range [{logits.min():.4f}, {logits.max():.4f}]",
        f"probability range [{probabilities.min():.4f}, {probabilities.max():.4f}]"
    )


    for i in range(len(X)):

        rows.append({
            "seed": seed,
            "image_id": str(image_ids[i]),
            "original_grade": int(grades[i]),
            "binary_label": int(labels[i]),
            "raw_logit": float(logits[i]),
            "raw_probability": float(probabilities[i])
        })


# ============================================================
# RESULT
# ============================================================

predictions = pd.DataFrame(rows)

print("\n========== RESULT ==========\n")

print("Prediction rows:", len(predictions))
print("Expected:", 1740 * 5)

print(
    "NaN values:",
    predictions[
        ["raw_logit", "raw_probability"]
    ].isna().sum().sum()
)

print(
    "Seeds:",
    sorted(predictions["seed"].unique())
)

assert len(predictions) == 1740 * 5
assert predictions["raw_logit"].isna().sum() == 0
assert predictions["raw_probability"].isna().sum() == 0


# ============================================================
# SAVE
# ============================================================

predictions.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\n✅ Saved:")
print(OUTPUT_PATH)

print("\nNO TRAINING WAS PERFORMED.")
print("========== COMPLETE ==========")

Images: 1740
Embedding shape: (1740, 384)
Heads: dict_keys([42, 43, 44, 45, 46])
Seed 42: logit range [-13.5696, 9.2100] probability range [0.0000, 0.9999]
Seed 43: logit range [-13.5779, 7.6045] probability range [0.0000, 0.9995]
Seed 44: logit range [-11.1823, 7.3431] probability range [0.0000, 0.9994]
Seed 45: logit range [-12.3095, 6.8802] probability range [0.0000, 0.9990]
Seed 46: logit range [-11.0764, 7.6582] probability range [0.0000, 0.9995]

========== RESULT ==========

Prediction rows: 8700
Expected: 8700
NaN values: 0
Seeds: [np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46)]

✅ Saved:
/content/drive/MyDrive/thesis_DR/messidor2_retfound_raw_predictions.csv

NO TRAINING WAS PERFORMED.
========== COMPLETE ==========


**Messidor-2 External Evaluation**

In [12]:
# ============================================================
# MESSIDOR-2 EXTERNAL EVALUATION
# Frozen APTOS temperature + frozen 5% LTT policy
# ============================================================

import json
import numpy as np
import pandas as pd

from pathlib import Path
from scipy.special import expit
from scipy.stats import beta

PROJECT_DIR = Path("/content/drive/MyDrive/thesis_DR")

PRED_PATH = PROJECT_DIR / "messidor2_retfound_raw_predictions.csv"
TEMP_PATH = PROJECT_DIR / "aptos_retfound_temperature_scaling.json"
LTT_PATH = PROJECT_DIR / "aptos_retfound_ltt_risk_control.csv"

OUTPUT_PATH = PROJECT_DIR / "messidor2_external_results.csv"

SEEDS = [42, 43, 44, 45, 46]
ALPHA = 0.05


# ============================================================
# LOAD
# ============================================================

pred = pd.read_csv(PRED_PATH)
ltt = pd.read_csv(LTT_PATH)

with open(TEMP_PATH, "r") as f:
    temp_data = json.load(f)


# ============================================================
# 95% ONE-SIDED EXACT UPPER BOUND
# ============================================================

def upper_risk_bound(errors, n):

    if n == 0:
        return np.nan

    if errors == n:
        return 1.0

    return float(
        beta.ppf(
            0.95,
            errors + 1,
            n - errors
        )
    )


# ============================================================
# EVALUATE EACH FROZEN SOURCE POLICY
# ============================================================

rows = []

print("========== MESSIDOR-2 EXTERNAL VALIDATION ==========\n")

for seed in SEEDS:

    df = pred[
        pred["seed"] == seed
    ].copy()

    assert len(df) == 1740

    y = df["binary_label"].to_numpy(dtype=int)
    logits = df["raw_logit"].to_numpy(dtype=float)

    # --------------------------------------------------------
    # Frozen APTOS temperature
    # --------------------------------------------------------

    T = float(
        temp_data["seeds"][str(seed)]["temperature"]
    )

    calibrated_prob = expit(
        logits / T
    )

    # --------------------------------------------------------
    # Frozen APTOS 5% LTT threshold
    # --------------------------------------------------------

    policy = ltt[
        (ltt["seed"] == seed) &
        np.isclose(ltt["alpha"], ALPHA)
    ]

    assert len(policy) == 1

    threshold = float(
        policy.iloc[0]["selected_threshold"]
    )

    # --------------------------------------------------------
    # External auto-clear
    # --------------------------------------------------------

    auto_clear = calibrated_prob <= threshold

    total = len(y)
    referable_total = int((y == 1).sum())
    nonreferable_total = int((y == 0).sum())

    auto_clear_count = int(auto_clear.sum())

    errors = int(
        (
            auto_clear &
            (y == 1)
        ).sum()
    )

    coverage = auto_clear_count / total
    review_burden = 1 - coverage

    if auto_clear_count > 0:
        risk = errors / auto_clear_count
        npv = 1 - risk
        upper = upper_risk_bound(
            errors,
            auto_clear_count
        )
    else:
        risk = np.nan
        npv = np.nan
        upper = np.nan

    referable_miss = (
        errors / referable_total
        if referable_total > 0
        else np.nan
    )

    rows.append({
        "seed": seed,
        "temperature": T,
        "frozen_threshold": threshold,
        "total_cases": total,
        "non_referable": nonreferable_total,
        "referable": referable_total,
        "auto_clear_count": auto_clear_count,
        "referable_auto_clear_errors": errors,
        "coverage": coverage,
        "review_burden": review_burden,
        "auto_clear_risk": risk,
        "auto_clear_npv": npv,
        "referable_miss_proportion": referable_miss,
        "upper_95_risk_bound": upper,
        "observed_risk_le_5pct": risk <= ALPHA,
        "upper_bound_le_5pct": upper <= ALPHA
    })

    print(f"---------- SEED {seed} ----------")
    print("Temperature:", f"{T:.6f}")
    print("Frozen threshold:", f"{threshold:.6f}")
    print("Cases:", total)
    print("Referable:", referable_total)
    print("Non-referable:", nonreferable_total)
    print("Auto-clear:", auto_clear_count)
    print("Referable errors:", errors)
    print("Coverage:", f"{coverage:.4f}")
    print("Review burden:", f"{review_burden:.4f}")
    print("Auto-clear risk:", f"{risk:.4f}")
    print("Auto-clear NPV:", f"{npv:.4f}")
    print("Referable miss proportion:", f"{referable_miss:.4f}")
    print("95% upper risk bound:", f"{upper:.4f}")
    print(
        "Observed risk <= 5%:",
        "YES" if risk <= ALPHA else "NO"
    )
    print(
        "95% upper bound <= 5%:",
        "YES" if upper <= ALPHA else "NO"
    )
    print()


# ============================================================
# SUMMARY
# ============================================================

results = pd.DataFrame(rows)

print("========== EXTERNAL SUMMARY ==========\n")

print(
    "Mean coverage:",
    f"{results['coverage'].mean():.4f}"
)

print(
    "Min coverage:",
    f"{results['coverage'].min():.4f}"
)

print(
    "Max coverage:",
    f"{results['coverage'].max():.4f}"
)

print(
    "Mean auto-clear risk:",
    f"{results['auto_clear_risk'].mean():.4f}"
)

print(
    "Seeds with observed risk <= 5%:",
    f"{results['observed_risk_le_5pct'].sum()} / 5"
)

print(
    "Seeds with 95% upper bound <= 5%:",
    f"{results['upper_bound_le_5pct'].sum()} / 5"
)


# ============================================================
# SAVE ONE RESULT FILE
# ============================================================

results.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\n✅ Saved:")
print(OUTPUT_PATH)

print("\nNO RETRAINING.")
print("NO RECALIBRATION.")
print("NO THRESHOLD CHANGES.")
print("========== COMPLETE ==========")

========== MESSIDOR-2 EXTERNAL VALIDATION ==========

---------- SEED 42 ----------
Temperature: 1.144044
Frozen threshold: 0.159284
Cases: 1740
Referable: 455
Non-referable: 1285
Auto-clear: 1446
Referable errors: 236
Coverage: 0.8310
Review burden: 0.1690
Auto-clear risk: 0.1632
Auto-clear NPV: 0.8368
Referable miss proportion: 0.5187
95% upper risk bound: 0.1800
Observed risk <= 5%: NO
95% upper bound <= 5%: NO

---------- SEED 43 ----------
Temperature: 1.002538
Frozen threshold: 0.027966
Cases: 1740
Referable: 455
Non-referable: 1285
Auto-clear: 1333
Referable errors: 182
Coverage: 0.7661
Review burden: 0.2339
Auto-clear risk: 0.1365
Auto-clear NPV: 0.8635
Referable miss proportion: 0.4000
95% upper risk bound: 0.1530
Observed risk <= 5%: NO
95% upper bound <= 5%: NO

---------- SEED 44 ----------
Temperature: 1.273578
Frozen threshold: 0.046259
Cases: 1740
Referable: 455
Non-referable: 1285
Auto-clear: 1314
Referable errors: 180
Coverage: 0.7552
Review burden: 0.2448
Auto-clear r

**Messidor-2 Evaluation Metrics**

In [13]:
# ============================================================
# MESSIDOR-2 EVALUATION METRICS
# DISPLAY ONLY - NOTHING SAVED
# ============================================================

import json
import numpy as np
import pandas as pd

from pathlib import Path
from scipy.special import expit
from scipy.stats import beta

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    balanced_accuracy_score,
    log_loss,
    brier_score_loss
)

# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/thesis_DR")

PRED_PATH = PROJECT_DIR / "messidor2_retfound_raw_predictions.csv"
TEMP_PATH = PROJECT_DIR / "aptos_retfound_temperature_scaling.json"
LTT_PATH = PROJECT_DIR / "aptos_retfound_ltt_risk_control.csv"

SEEDS = [42, 43, 44, 45, 46]
ALPHA = 0.05
ECE_BINS = 10


# ============================================================
# LOAD
# ============================================================

pred = pd.read_csv(PRED_PATH)
ltt = pd.read_csv(LTT_PATH)

with open(TEMP_PATH, "r") as f:
    temp_data = json.load(f)


# ============================================================
# ECE
# ============================================================

def calculate_ece(probs, labels, n_bins=10):

    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):

        if i == n_bins - 1:
            mask = (
                (probs >= bins[i]) &
                (probs <= bins[i + 1])
            )
        else:
            mask = (
                (probs >= bins[i]) &
                (probs < bins[i + 1])
            )

        if mask.sum() == 0:
            continue

        confidence = probs[mask].mean()
        accuracy = labels[mask].mean()

        ece += mask.mean() * abs(
            confidence - accuracy
        )

    return float(ece)


# ============================================================
# ONE-SIDED 95% EXACT UPPER RISK BOUND
# ============================================================

def upper_risk_bound(errors, n):

    if n == 0:
        return np.nan

    if errors == n:
        return 1.0

    return float(
        beta.ppf(
            0.95,
            errors + 1,
            n - errors
        )
    )


classification_rows = []
calibration_rows = []
selective_rows = []


# ============================================================
# EACH SEED
# ============================================================

for seed in SEEDS:

    df = pred[
        pred["seed"] == seed
    ].copy()

    assert len(df) == 1740

    y = df["binary_label"].to_numpy(dtype=int)
    logits = df["raw_logit"].to_numpy(dtype=float)

    # Frozen APTOS temperature
    T = float(
        temp_data["seeds"][str(seed)]["temperature"]
    )

    raw_prob = expit(logits)
    calibrated_prob = expit(logits / T)


    # ========================================================
    # 1. CLASSIFICATION METRICS
    # ========================================================

    y_pred = (
        calibrated_prob >= 0.5
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y,
        y_pred,
        labels=[0, 1]
    ).ravel()

    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)

    ppv = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else np.nan
    )

    npv = (
        tn / (tn + fn)
        if (tn + fn) > 0
        else np.nan
    )

    classification_rows.append({

        "Seed": seed,

        "AUROC":
            roc_auc_score(
                y,
                calibrated_prob
            ),

        "AUPRC":
            average_precision_score(
                y,
                calibrated_prob
            ),

        "Sensitivity":
            sensitivity,

        "Specificity":
            specificity,

        "PPV":
            ppv,

        "NPV":
            npv,

        "F1":
            f1_score(
                y,
                y_pred
            ),

        "Balanced Accuracy":
            balanced_accuracy_score(
                y,
                y_pred
            )
    })


    # ========================================================
    # 2. CALIBRATION METRICS
    # ========================================================

    calibration_rows.append({

        "Seed": seed,

        "Raw NLL":
            log_loss(
                y,
                raw_prob,
                labels=[0, 1]
            ),

        "Calibrated NLL":
            log_loss(
                y,
                calibrated_prob,
                labels=[0, 1]
            ),

        "Raw Brier":
            brier_score_loss(
                y,
                raw_prob
            ),

        "Calibrated Brier":
            brier_score_loss(
                y,
                calibrated_prob
            ),

        "Raw ECE":
            calculate_ece(
                raw_prob,
                y,
                ECE_BINS
            ),

        "Calibrated ECE":
            calculate_ece(
                calibrated_prob,
                y,
                ECE_BINS
            )
    })


    # ========================================================
    # 3. FROZEN APTOS 5% LTT POLICY
    # ========================================================

    policy = ltt[
        (ltt["seed"] == seed) &
        np.isclose(
            ltt["alpha"],
            ALPHA
        )
    ].iloc[0]

    threshold = float(
        policy["selected_threshold"]
    )

    auto_clear = (
        calibrated_prob <= threshold
    )

    auto_clear_count = int(
        auto_clear.sum()
    )

    errors = int(
        (
            auto_clear &
            (y == 1)
        ).sum()
    )

    referable_total = int(
        (y == 1).sum()
    )

    coverage = (
        auto_clear_count / len(y)
    )

    risk = (
        errors / auto_clear_count
        if auto_clear_count > 0
        else np.nan
    )

    miss_proportion = (
        errors / referable_total
        if referable_total > 0
        else np.nan
    )

    upper = upper_risk_bound(
        errors,
        auto_clear_count
    )

    selective_rows.append({

        "Seed": seed,

        "Frozen Threshold":
            threshold,

        "Auto-clear Count":
            auto_clear_count,

        "Referable Errors":
            errors,

        "Coverage":
            coverage,

        "Review Burden":
            1 - coverage,

        "Auto-clear Risk":
            risk,

        "Auto-clear NPV":
            1 - risk,

        "Referable Miss Proportion":
            miss_proportion,

        "95% Upper Risk Bound":
            upper,

        "Risk Target Met":
            upper <= ALPHA
    })


# ============================================================
# DATAFRAMES
# ============================================================

classification_df = pd.DataFrame(
    classification_rows
)

calibration_df = pd.DataFrame(
    calibration_rows
)

selective_df = pd.DataFrame(
    selective_rows
)


# ============================================================
# MEAN + STD
# ============================================================

def add_summary(df):

    numeric_cols = df.select_dtypes(
        include=np.number
    ).columns.drop(
        "Seed",
        errors="ignore"
    )

    mean_row = {"Seed": "MEAN"}
    std_row = {"Seed": "STD"}

    for col in numeric_cols:
        mean_row[col] = df[col].mean()
        std_row[col] = df[col].std(ddof=1)

    return pd.concat(
        [
            df,
            pd.DataFrame(
                [mean_row, std_row]
            )
        ],
        ignore_index=True
    )


classification_show = add_summary(
    classification_df
)

calibration_show = add_summary(
    calibration_df
)

selective_show = add_summary(
    selective_df.drop(
        columns=["Risk Target Met"]
    )
)


# ============================================================
# DISPLAY
# ============================================================

pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.width",
    200
)

pd.set_option(
    "display.float_format",
    lambda x: f"{x:.4f}"
)


print(
    "\n========== 1. CLASSIFICATION PERFORMANCE ==========\n"
)

print("Classification threshold = 0.5\n")

display(classification_show)


print(
    "\n========== 2. CALIBRATION PERFORMANCE ==========\n"
)

display(calibration_show)


print(
    "\n========== 3. SELECTIVE RISK-CONTROL PERFORMANCE ==========\n"
)

display(selective_show)


print(
    "\n95% risk target satisfied:",
    f"{selective_df['Risk Target Met'].sum()} / 5 seeds"
)

print(
    "\nDISPLAY ONLY - NO FILE WAS SAVED."
)


========== 1. CLASSIFICATION PERFORMANCE ==========

Classification threshold = 0.5



,Seed,AUROC,AUPRC,Sensitivity,Specificity,PPV,NPV,F1,Balanced Accuracy
0,42,0.8265,0.6959,0.3473,0.9821,0.8729,0.8095,0.4969,0.6647
1,43,0.8364,0.7172,0.2923,0.9953,0.9568,0.7989,0.4478,0.6438
2,44,0.8267,0.7010,0.2571,0.9930,0.9286,0.7906,0.4028,0.6251
3,45,0.8236,0.6999,0.2769,0.9907,0.9130,0.7946,0.4250,0.6338
4,46,0.8331,0.7059,0.3538,0.9860,0.8994,0.8117,0.5079,0.6699
5,MEAN,0.8293,0.7040,0.3055,0.9894,0.9142,0.8010,0.4561,0.6475
6,STD,0.0053,0.0082,0.0430,0.0054,0.0314,0.0092,0.0454,0.0194



========== 2. CALIBRATION PERFORMANCE ==========



,Seed,Raw NLL,Calibrated NLL,Raw Brier,Calibrated Brier,Raw ECE,Calibrated ECE
0,42,0.8437,0.7516,0.1637,0.1607,0.1510,0.1449
1,43,0.8756,0.8737,0.1685,0.1685,0.1714,0.1714
2,44,0.8808,0.7159,0.1766,0.1703,0.1773,0.1689
3,45,0.8479,0.8013,0.1723,0.1708,0.1707,0.1686
4,46,0.6994,0.6581,0.1564,0.1546,0.1438,0.1419
5,MEAN,0.8295,0.7601,0.1675,0.1650,0.1629,0.1591
6,STD,0.0746,0.0822,0.0078,0.0071,0.0146,0.0145



========== 3. SELECTIVE RISK-CONTROL PERFORMANCE ==========



,Seed,Frozen Threshold,Auto-clear Count,Referable Errors,Coverage,Review Burden,Auto-clear Risk,Auto-clear NPV,Referable Miss Proportion,95% Upper Risk Bound
0,42,0.1593,1446.0000,236.0000,0.8310,0.1690,0.1632,0.8368,0.5187,0.1800
1,43,0.0280,1333.0000,182.0000,0.7661,0.2339,0.1365,0.8635,0.4000,0.1530
2,44,0.0463,1314.0000,180.0000,0.7552,0.2448,0.1370,0.8630,0.3956,0.1536
3,45,0.0747,1411.0000,221.0000,0.8109,0.1891,0.1566,0.8434,0.4857,0.1734
4,46,0.0972,1346.0000,188.0000,0.7736,0.2264,0.1397,0.8603,0.4132,0.1562
5,MEAN,0.0811,1370.0000,201.4000,0.7874,0.2126,0.1466,0.8534,0.4426,0.1632
6,STD,0.0511,55.9866,25.4716,0.0322,0.0322,0.0124,0.0124,0.0560,0.0126



95% risk target satisfied: 0 / 5 seeds

DISPLAY ONLY - NO FILE WAS SAVED.


#**Messidor-2 Re-certification Split**

In [9]:
# ============================================================
# MESSIDOR-2 LOCAL RE-CERTIFICATION SPLIT
# 10% probability calibration
# 20% risk calibration
# 70% untouched evaluation
# ============================================================

import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import train_test_split


# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/thesis_DR")

EMBED_PATH = (
    PROJECT_DIR /
    "messidor2_retfound_green_embeddings.npz"
)

OUTPUT_PATH = (
    PROJECT_DIR /
    "messidor2_recert_split_manifest.csv"
)

RANDOM_SEED = 2026


# ============================================================
# LOAD
# ============================================================

data = np.load(
    EMBED_PATH,
    allow_pickle=True
)

df = pd.DataFrame({

    "image_id":
        data["image_id"].astype(str),

    "original_grade":
        data["original_grade"].astype(int),

    "binary_label":
        data["binary_label"].astype(int)
})


assert len(df) == 1740
assert df["image_id"].nunique() == 1740


# ============================================================
# FIRST SPLIT
#
# 30% adaptation
# 70% untouched evaluation
# ============================================================

adapt, evaluation = train_test_split(

    df,

    test_size=0.70,

    random_state=RANDOM_SEED,

    stratify=df["original_grade"]
)


# ============================================================
# SECOND SPLIT
#
# Adaptation = 522 images
#
# 1/3 of adaptation = 174 = probability calibration
# 2/3 of adaptation = 348 = risk calibration
# ============================================================

prob_cal, risk_cal = train_test_split(

    adapt,

    test_size=2/3,

    random_state=RANDOM_SEED + 1,

    stratify=adapt["original_grade"]
)


# ============================================================
# ASSIGN SPLIT NAMES
# ============================================================

prob_cal = prob_cal.copy()
risk_cal = risk_cal.copy()
evaluation = evaluation.copy()

prob_cal["split"] = "prob_calibration"
risk_cal["split"] = "risk_calibration"
evaluation["split"] = "evaluation"


manifest = pd.concat(
    [
        prob_cal,
        risk_cal,
        evaluation
    ],
    ignore_index=True
)


# ============================================================
# CHECK
# ============================================================

print(
    "========== MESSIDOR-2 RE-CERTIFICATION SPLIT ==========\n"
)

summary = (
    manifest
    .groupby("split")
    .agg(
        total=("image_id", "count"),
        non_referable=(
            "binary_label",
            lambda x: int((x == 0).sum())
        ),
        referable=(
            "binary_label",
            lambda x: int((x == 1).sum())
        )
    )
)

display(summary)


print(
    "\nTotal:",
    len(manifest)
)

print(
    "Unique images:",
    manifest["image_id"].nunique()
)


assert len(manifest) == 1740
assert manifest["image_id"].nunique() == 1740

assert (
    (manifest["split"] == "prob_calibration").sum()
    == 174
)

assert (
    (manifest["split"] == "risk_calibration").sum()
    == 348
)

assert (
    (manifest["split"] == "evaluation").sum()
    == 1218
)


# ============================================================
# SAVE THE LOCKED SPLIT
# ============================================================

manifest.to_csv(
    OUTPUT_PATH,
    index=False
)

print(
    "\n✅ Saved permanently:"
)

print(
    OUTPUT_PATH
)

print(
    "\nThis split is now LOCKED."
)

print(
    "The 1218 evaluation images must not be used "
    "for calibration or threshold selection."
)

========== MESSIDOR-2 RE-CERTIFICATION SPLIT ==========



,total,non_referable,referable
split,,,
evaluation,1218,900,318
prob_calibration,174,129,45
risk_calibration,348,256,92



Total: 1740
Unique images: 1740

✅ Saved permanently:
/content/drive/MyDrive/thesis_DR/messidor2_recert_split_manifest.csv

This split is now LOCKED.
The 1218 evaluation images must not be used for calibration or threshold selection.


#Messidor-2 Local Temperature Scaling

In [10]:
# ============================================================
# MESSIDOR-2 LOCAL TEMPERATURE SCALING
# Uses ONLY 174 probability-calibration images
# ============================================================

import json
import numpy as np
import pandas as pd

from pathlib import Path
from scipy.special import expit
from scipy.optimize import minimize_scalar
from sklearn.metrics import log_loss, brier_score_loss


# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/thesis_DR")

PRED_PATH = PROJECT_DIR / "messidor2_retfound_raw_predictions.csv"
SPLIT_PATH = PROJECT_DIR / "messidor2_recert_split_manifest.csv"

OUTPUT_PATH = PROJECT_DIR / "messidor2_local_temperature_scaling.json"

SEEDS = [42, 43, 44, 45, 46]


# ============================================================
# LOAD
# ============================================================

pred = pd.read_csv(
    PRED_PATH,
    dtype={"image_id": str}
)

manifest = pd.read_csv(
    SPLIT_PATH,
    dtype={"image_id": str}
)

prob_ids = set(
    manifest.loc[
        manifest["split"] == "prob_calibration",
        "image_id"
    ]
)

assert len(prob_ids) == 174


# ============================================================
# FIT TEMPERATURE
# ============================================================

def fit_temperature(logits, labels):

    logits = np.asarray(logits, dtype=float)
    labels = np.asarray(labels, dtype=int)

    def objective(log_T):

        T = np.exp(log_T)

        probs = expit(
            logits / T
        )

        return log_loss(
            labels,
            probs,
            labels=[0, 1]
        )

    result = minimize_scalar(
        objective,
        bounds=(np.log(0.05), np.log(20.0)),
        method="bounded"
    )

    return float(
        np.exp(result.x)
    )


# ============================================================
# EACH FROZEN APTOS HEAD
# ============================================================

results = {
    "dataset": "MESSIDOR-2",
    "probability_calibration_images": 174,
    "seeds": {}
}

print(
    "========== MESSIDOR-2 LOCAL TEMPERATURE SCALING ==========\n"
)

for seed in SEEDS:

    df = pred[
        (pred["seed"] == seed) &
        (pred["image_id"].isin(prob_ids))
    ].copy()

    assert len(df) == 174

    y = df[
        "binary_label"
    ].to_numpy(dtype=int)

    logits = df[
        "raw_logit"
    ].to_numpy(dtype=float)

    # Before scaling
    raw_prob = expit(logits)

    raw_nll = log_loss(
        y,
        raw_prob,
        labels=[0, 1]
    )

    raw_brier = brier_score_loss(
        y,
        raw_prob
    )

    # Fit target temperature
    T = fit_temperature(
        logits,
        y
    )

    # After scaling
    calibrated_prob = expit(
        logits / T
    )

    calibrated_nll = log_loss(
        y,
        calibrated_prob,
        labels=[0, 1]
    )

    calibrated_brier = brier_score_loss(
        y,
        calibrated_prob
    )

    results["seeds"][str(seed)] = {
        "temperature": T,
        "raw_nll": raw_nll,
        "calibrated_nll": calibrated_nll,
        "raw_brier": raw_brier,
        "calibrated_brier": calibrated_brier
    }

    print(f"---------- SEED {seed} ----------")
    print(f"Temperature : {T:.6f}")
    print(
        f"NLL         : "
        f"{raw_nll:.4f} -> {calibrated_nll:.4f}"
    )
    print(
        f"Brier       : "
        f"{raw_brier:.4f} -> {calibrated_brier:.4f}"
    )
    print()


# ============================================================
# SAVE ONE SMALL JSON
# ============================================================

with open(
    OUTPUT_PATH,
    "w"
) as f:
    json.dump(
        results,
        f,
        indent=2
    )

print("✅ Saved:")
print(OUTPUT_PATH)

print("\nOnly 174 calibration images were used.")
print("348 risk-calibration images remain untouched.")
print("1218 evaluation images remain untouched.")
print("========== COMPLETE ==========")

========== MESSIDOR-2 LOCAL TEMPERATURE SCALING ==========

---------- SEED 42 ----------
Temperature : 3.169531
NLL         : 0.6672 -> 0.3776
Brier       : 0.1454 -> 0.1191

---------- SEED 43 ----------
Temperature : 3.327310
NLL         : 0.7064 -> 0.3833
Brier       : 0.1558 -> 0.1223

---------- SEED 44 ----------
Temperature : 3.275613
NLL         : 0.7405 -> 0.4031
Brier       : 0.1649 -> 0.1293

---------- SEED 45 ----------
Temperature : 3.174622
NLL         : 0.7015 -> 0.3968
Brier       : 0.1622 -> 0.1273

---------- SEED 46 ----------
Temperature : 2.585638
NLL         : 0.5577 -> 0.3734
Brier       : 0.1400 -> 0.1188

✅ Saved:
/content/drive/MyDrive/thesis_DR/messidor2_local_temperature_scaling.json

Only 174 calibration images were used.
348 risk-calibration images remain untouched.
1218 evaluation images remain untouched.
========== COMPLETE ==========


#**Direct Auto-clear Risk Certification**

#**IDRiD Download + Exact Audit**

In [16]:
# ============================================================
# DOWNLOAD FULL IDRiD DATASET + VERIFY
# ============================================================

!pip -q install kagglehub

from pathlib import Path
import kagglehub


# ============================================================
# DOWNLOAD
# ============================================================

path = Path(
    kagglehub.dataset_download(
        "dankok/diabetic-retinopathy-image-dataset"
    )
)

print("Downloaded to:")
print(path)


# ============================================================
# SHOW CSV FILES
# ============================================================

print("\n========== CSV FILES ==========\n")

csv_files = sorted(
    path.rglob("*.csv")
)

for p in csv_files:
    print(p)


# ============================================================
# FIND IDRiD JPG IMAGES
# ============================================================

jpg_files = sorted(
    [
        p for p in path.rglob("*")
        if p.suffix.lower() in {".jpg", ".jpeg"}
        and "IDRiD" in p.name
    ]
)

print(
    "\n========== IDRiD JPG FILES ==========\n"
)

print(
    "Total IDRiD JPG files found:",
    len(jpg_files)
)


print(
    "\nFirst 15:"
)

for p in jpg_files[:15]:
    print(p)


# ============================================================
# SHOW MAIN DIRECTORY STRUCTURE
# ============================================================

print(
    "\n========== MAIN FOLDERS ==========\n"
)

for p in sorted(path.iterdir()):
    print(p)

100%|██████████| 408M/408M [00:24<00:00, 17.2MB/s]

Extracting files...


Downloaded to:
/root/.cache/kagglehub/datasets/dankok/diabetic-retinopathy-image-dataset/versions/1

========== CSV FILES ==========

/root/.cache/kagglehub/datasets/dankok/diabetic-retinopathy-image-dataset/versions/1/Disease_Grading/Groundtruths/IDRiD_Disease Grading_Testing Labels.csv
/root/.cache/kagglehub/datasets/dankok/diabetic-retinopathy-image-dataset/versions/1/Disease_Grading/Groundtruths/IDRiD_Disease Grading_Training Labels.csv
/root/.cache/kagglehub/datasets/dankok/diabetic-retinopathy-image-dataset/versions/1/Localization/Localization/Groundtruths/Fovea Center Location/IDRiD_Fovea_Center_Testing Set_Markups.csv
/root/.cache/kagglehub/datasets/dankok/diabetic-retinopathy-image-dataset/versions/1/Localization/Localization/Groundtruths/Fovea Center Location/IDRiD_Fovea_Center_Training Set_Markups.csv
/root/.cache/kagglehub/datasets/dankok/diabetic-retinopathy-image-dataset/versions/1/Localization/Localization/Groundtruths/Optic Disc Center Location/IDRiD_OD_Center_Testing S

In [18]:
# ============================================================
# IDRiD DISEASE-GRADING EXACT DUPLICATE AUDIT
# FIXED VERSION
# ============================================================

import hashlib
from pathlib import Path
import pandas as pd


# ============================================================
# PATHS
# ============================================================

BASE = Path(
    "/root/.cache/kagglehub/datasets/"
    "dankok/diabetic-retinopathy-image-dataset/versions/1"
)

GRADING = BASE / "Disease_Grading"

TRAIN_IMG_DIR = GRADING / "Original_Images" / "Training Set"
TEST_IMG_DIR  = GRADING / "Original_Images" / "Testing Set"

TRAIN_CSV = (
    GRADING
    / "Groundtruths"
    / "IDRiD_Disease Grading_Training Labels.csv"
)

TEST_CSV = (
    GRADING
    / "Groundtruths"
    / "IDRiD_Disease Grading_Testing Labels.csv"
)

PROJECT_DIR = Path("/content/drive/MyDrive/thesis_DR")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = PROJECT_DIR / "idrid_exact_duplicate_audit.csv"


# ============================================================
# LOAD LABELS
# ============================================================

train_labels = pd.read_csv(TRAIN_CSV)
test_labels = pd.read_csv(TEST_CSV)

train_labels = train_labels[
    ["Image name", "Retinopathy grade"]
].copy()

test_labels = test_labels[
    ["Image name", "Retinopathy grade"]
].copy()


train_labels.columns = [
    "original_image_id",
    "original_grade"
]

test_labels.columns = [
    "original_image_id",
    "original_grade"
]


train_labels["source_split"] = "train"
test_labels["source_split"] = "test"


labels = pd.concat(
    [train_labels, test_labels],
    ignore_index=True
)


labels["original_image_id"] = (
    labels["original_image_id"]
    .astype(str)
    .str.strip()
)

labels["original_grade"] = (
    labels["original_grade"]
    .astype(int)
)


# ============================================================
# BASIC CHECK
# ============================================================

assert len(train_labels) == 413
assert len(test_labels) == 103
assert len(labels) == 516

assert set(
    labels["original_grade"].unique()
).issubset({0, 1, 2, 3, 4})


# ============================================================
# BUILD IMAGE MAP
# ============================================================

def build_image_map(folder):

    files = sorted(
        [
            p for p in folder.iterdir()
            if p.suffix.lower()
            in {".jpg", ".jpeg", ".png"}
        ]
    )

    return {
        p.stem.strip(): p
        for p in files
    }


train_map = build_image_map(TRAIN_IMG_DIR)
test_map = build_image_map(TEST_IMG_DIR)


print("========== IDRiD GRADING IMAGES ==========\n")

print("Training images:", len(train_map))
print("Testing images :", len(test_map))
print("Total          :", len(train_map) + len(test_map))


assert len(train_map) == 413
assert len(test_map) == 103


# ============================================================
# MATCH LABELS WITH IMAGES
# ============================================================

rows = []


for _, row in labels.iterrows():

    original_id = row["original_image_id"]
    split = row["source_split"]
    grade = int(row["original_grade"])

    image_map = (
        train_map
        if split == "train"
        else test_map
    )

    if original_id not in image_map:

        raise RuntimeError(
            f"Missing image: {split} / {original_id}"
        )


    # Globally unique ID
    image_id = f"{split}_{original_id}"


    binary_label = (
        0 if grade <= 1 else 1
    )


    rows.append({
        "image_id": image_id,
        "original_image_id": original_id,
        "source_split": split,
        "original_grade": grade,
        "binary_label": binary_label,
        "path": str(image_map[original_id])
    })


df = pd.DataFrame(rows)


# ============================================================
# IMPORTANT CHECK
# ============================================================

assert len(df) == 516
assert df["image_id"].nunique() == 516

print("\n✅ 516 unique disease-grading records confirmed")


# ============================================================
# SHA-256
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


print("\nCalculating SHA-256 hashes...")


df["sha256"] = [
    sha256_file(path)
    for path in df["path"]
]


# ============================================================
# EXACT DUPLICATE AUDIT
# ============================================================

df["audit_status"] = "KEEP"

duplicate_groups = 0
images_in_duplicate_groups = 0
same_grade_groups = 0
grade_conflict_groups = 0
binary_conflict_groups = 0


for sha, group in df.groupby("sha256"):

    if len(group) <= 1:
        continue


    duplicate_groups += 1
    images_in_duplicate_groups += len(group)


    grades = set(
        group["original_grade"].tolist()
    )

    binary_labels = set(
        group["binary_label"].tolist()
    )


    # ========================================================
    # EXACT IMAGE + CONFLICTING GRADE
    # Exclude whole group
    # ========================================================

    if len(grades) > 1:

        grade_conflict_groups += 1

        if len(binary_labels) > 1:
            binary_conflict_groups += 1


        df.loc[
            group.index,
            "audit_status"
        ] = "EXCLUDE_LABEL_CONFLICT"


    # ========================================================
    # EXACT IMAGE + SAME GRADE
    # Keep one copy
    # ========================================================

    else:

        same_grade_groups += 1


        sorted_group = group.sort_values(
            [
                "source_split",
                "original_image_id"
            ]
        )


        keep_index = sorted_group.index[0]

        drop_indices = [
            idx
            for idx in sorted_group.index
            if idx != keep_index
        ]


        df.loc[
            drop_indices,
            "audit_status"
        ] = "DROP_EXACT_DUPLICATE"


# ============================================================
# FINAL KEEP COHORT
# ============================================================

final_df = df[
    df["audit_status"] == "KEEP"
].copy()

final_df = final_df.sort_values(
    [
        "source_split",
        "original_image_id"
    ]
).reset_index(drop=True)


# ============================================================
# SUMMARY
# ============================================================

print("\n========== IDRiD EXACT AUDIT ==========\n")

print(
    "Raw disease-grading images :",
    len(df)
)

print(
    "Unique SHA-256 hashes      :",
    df["sha256"].nunique()
)

print(
    "Exact duplicate groups     :",
    duplicate_groups
)

print(
    "Images in duplicate groups :",
    images_in_duplicate_groups
)

print(
    "Same-grade groups          :",
    same_grade_groups
)

print(
    "Grade-conflict groups      :",
    grade_conflict_groups
)

print(
    "Binary-conflict groups     :",
    binary_conflict_groups
)


print("\nAudit decisions:")

print(
    df["audit_status"]
    .value_counts()
    .to_string()
)


print(
    "\nFinal KEEP cohort:",
    len(final_df)
)


# ============================================================
# ORIGINAL GRADE COUNTS
# ============================================================

print("\nOriginal grade counts:")

print(
    final_df["original_grade"]
    .value_counts()
    .sort_index()
    .to_string()
)


# ============================================================
# BINARY COUNTS
# ============================================================

non_ref = int(
    (final_df["binary_label"] == 0).sum()
)

ref = int(
    (final_df["binary_label"] == 1).sum()
)


print("\nBinary referable-DR counts:")

print(
    f"Non-referable (0-1): {non_ref}"
)

print(
    f"Referable (2-4)    : {ref}"
)


# ============================================================
# SAVE
# ============================================================

df.to_csv(
    OUTPUT_PATH,
    index=False
)


print("\n✅ Saved permanently:")
print(OUTPUT_PATH)

print("\n========== COMPLETE ==========")

========== IDRiD GRADING IMAGES ==========

Training images: 413
Testing images : 103
Total          : 516

✅ 516 unique disease-grading records confirmed

Calculating SHA-256 hashes...

========== IDRiD EXACT AUDIT ==========

Raw disease-grading images : 516
Unique SHA-256 hashes      : 510
Exact duplicate groups     : 6
Images in duplicate groups : 12
Same-grade groups          : 3
Grade-conflict groups      : 3
Binary-conflict groups     : 2

Audit decisions:
audit_status
KEEP                      507
EXCLUDE_LABEL_CONFLICT      6
DROP_EXACT_DUPLICATE        3

Final KEEP cohort: 507

Original grade counts:
original_grade
0    167
1     23
2    165
3     91
4     61

Binary referable-DR counts:
Non-referable (0-1): 190
Referable (2-4)    : 317

✅ Saved permanently:
/content/drive/MyDrive/thesis_DR/idrid_exact_duplicate_audit.csv

========== COMPLETE ==========


#**DRiD RETFound-Green Embeddings**

In [20]:
# ============================================================
# IDRiD RETFOUND-GREEN EMBEDDINGS - FIXED
# Final cohort: 507 KEEP images
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm


# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/thesis_DR")

AUDIT_PATH = PROJECT_DIR / "idrid_exact_duplicate_audit.csv"
CHECKPOINT_PATH = PROJECT_DIR / "retfoundgreen_statedict.pth"

OUTPUT_PATH = (
    PROJECT_DIR / "idrid_retfound_green_embeddings.npz"
)


# ============================================================
# LOAD FINAL IDRiD COHORT
# ============================================================

df = pd.read_csv(
    AUDIT_PATH,
    dtype={
        "image_id": str,
        "original_image_id": str
    }
)

df = df[
    df["audit_status"] == "KEEP"
].copy()

df = df.sort_values(
    [
        "source_split",
        "original_image_id"
    ]
).reset_index(drop=True)


assert len(df) == 507
assert df["image_id"].nunique() == 507


print("Final IDRiD cohort:", len(df))


# ============================================================
# LOCKED PREPROCESSING
# ============================================================

def remove_black_border(img):

    arr = np.asarray(img)

    # Remove only obvious exterior black border
    mask = arr.max(axis=2) > 10

    if not mask.any():
        return img

    ys, xs = np.where(mask)

    left = int(xs.min())
    right = int(xs.max()) + 1
    top = int(ys.min())
    bottom = int(ys.max()) + 1

    return img.crop(
        (left, top, right, bottom)
    )


def pad_square(img):

    width, height = img.size

    side = max(
        width,
        height
    )

    canvas = Image.new(
        "RGB",
        (side, side),
        (0, 0, 0)
    )

    x = (side - width) // 2
    y = (side - height) // 2

    canvas.paste(
        img,
        (x, y)
    )

    return canvas


to_tensor = transforms.ToTensor()

normalize = transforms.Normalize(
    mean=(0.5, 0.5, 0.5),
    std=(0.5, 0.5, 0.5)
)


def preprocess(img):

    img = img.convert("RGB")

    img = remove_black_border(img)

    img = pad_square(img)

    img = img.resize(
        (392, 392),
        Image.Resampling.BICUBIC
    )

    img = to_tensor(img)

    img = normalize(img)

    return img


# ============================================================
# DATASET
# ============================================================

class IDRiDDataset(Dataset):

    def __init__(self, dataframe):

        self.df = (
            dataframe
            .reset_index(drop=True)
        )

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        path = self.df.iloc[idx]["path"]

        with Image.open(path) as img:

            image = preprocess(
                img
            )

        return image


dataset = IDRiDDataset(
    df
)


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)


loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=(
        device.type == "cuda"
    )
)


# ============================================================
# CREATE RETFOUND-GREEN MODEL
# ============================================================

model = timm.create_model(
    "vit_small_patch14_reg4_dinov2",
    pretrained=False,
    img_size=392,
    num_classes=0,
    global_pool="avg"
)


# ============================================================
# LOAD CHECKPOINT
# ============================================================

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)


# ============================================================
# HANDLE POSSIBLE CHECKPOINT WRAPPER
# ============================================================

if (
    isinstance(checkpoint, dict)
    and "state_dict" in checkpoint
):

    state_dict = checkpoint[
        "state_dict"
    ]

elif (
    isinstance(checkpoint, dict)
    and "model" in checkpoint
):

    state_dict = checkpoint[
        "model"
    ]

else:

    state_dict = checkpoint


# ============================================================
# CLEAN KEY PREFIXES
# ============================================================

clean_state = {}

for key, value in state_dict.items():

    new_key = key

    if new_key.startswith(
        "module."
    ):
        new_key = new_key[
            len("module.") :
        ]

    if new_key.startswith(
        "model."
    ):
        new_key = new_key[
            len("model.") :
        ]

    clean_state[
        new_key
    ] = value


# ============================================================
# TIMM COMPATIBILITY FIX
#
# Saved RETFound checkpoint:
#   norm.weight
#   norm.bias
#
# Current timm model:
#   fc_norm.weight
#   fc_norm.bias
# ============================================================

model_keys = set(
    model.state_dict().keys()
)


if (
    "norm.weight" in clean_state
    and
    "fc_norm.weight" in model_keys
):

    clean_state[
        "fc_norm.weight"
    ] = clean_state.pop(
        "norm.weight"
    )


if (
    "norm.bias" in clean_state
    and
    "fc_norm.bias" in model_keys
):

    clean_state[
        "fc_norm.bias"
    ] = clean_state.pop(
        "norm.bias"
    )


# ============================================================
# LOAD STRICTLY
# ============================================================

model.load_state_dict(
    clean_state,
    strict=True
)

print(
    "✅ RETFound-Green checkpoint loaded"
)


# ============================================================
# FREEZE ENCODER
# ============================================================

model = model.to(
    device
)

model.eval()

for parameter in model.parameters():

    parameter.requires_grad = False


trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    "Trainable parameters:",
    trainable
)

assert trainable == 0


# ============================================================
# EXTRACT EMBEDDINGS
# ============================================================

all_embeddings = []

print(
    "\nExtracting IDRiD embeddings..."
)


with torch.no_grad():

    for batch in loader:

        batch = batch.to(
            device,
            non_blocking=True
        )

        embeddings = model(
            batch
        )

        all_embeddings.append(
            embeddings
            .cpu()
            .numpy()
        )


embeddings = np.concatenate(
    all_embeddings,
    axis=0
)


# ============================================================
# VALIDATE
# ============================================================

assert embeddings.shape == (
    507,
    384
)

assert np.isfinite(
    embeddings
).all()


print(
    "\nEmbedding shape:",
    embeddings.shape
)

print(
    "NaN:",
    np.isnan(
        embeddings
    ).sum()
)

print(
    "Inf:",
    np.isinf(
        embeddings
    ).sum()
)


# ============================================================
# SAVE
# ============================================================

np.savez_compressed(

    OUTPUT_PATH,

    embeddings=embeddings,

    image_id=df[
        "image_id"
    ].astype(str).to_numpy(),

    original_image_id=df[
        "original_image_id"
    ].astype(str).to_numpy(),

    source_split=df[
        "source_split"
    ].astype(str).to_numpy(),

    original_grade=df[
        "original_grade"
    ].to_numpy(dtype=int),

    binary_label=df[
        "binary_label"
    ].to_numpy(dtype=int)
)


print(
    "\n✅ Saved permanently:"
)

print(
    OUTPUT_PATH
)

print(
    "\n========== COMPLETE =========="
)

Final IDRiD cohort: 507
Device: cuda
✅ RETFound-Green checkpoint loaded
Trainable parameters: 0

Extracting IDRiD embeddings...

Embedding shape: (507, 384)
NaN: 0
Inf: 0

✅ Saved permanently:
/content/drive/MyDrive/thesis_DR/idrid_retfound_green_embeddings.npz

========== COMPLETE ==========


#**IDRiD Frozen APTOS MLP Predictions**

In [21]:
# ============================================================
# IDRiD - FROZEN APTOS MLP PREDICTIONS
# 507 IDRiD images x 5 APTOS heads
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn.functional as F


# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/thesis_DR"
)

EMBED_PATH = (
    PROJECT_DIR
    / "idrid_retfound_green_embeddings.npz"
)

HEADS_PATH = (
    PROJECT_DIR
    / "aptos_retfound_mlp_heads.pt"
)

OUTPUT_PATH = (
    PROJECT_DIR
    / "idrid_retfound_raw_predictions.csv"
)

SEEDS = [42, 43, 44, 45, 46]


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)


# ============================================================
# LOAD IDRiD EMBEDDINGS
# ============================================================

data = np.load(
    EMBED_PATH,
    allow_pickle=True
)

X = data["embeddings"].astype(
    np.float32
)

image_ids = (
    data["image_id"]
    .astype(str)
)

original_image_ids = (
    data["original_image_id"]
    .astype(str)
)

source_split = (
    data["source_split"]
    .astype(str)
)

original_grade = (
    data["original_grade"]
    .astype(int)
)

binary_label = (
    data["binary_label"]
    .astype(int)
)


# ============================================================
# VALIDATE
# ============================================================

assert X.shape == (507, 384)

assert len(image_ids) == 507
assert len(np.unique(image_ids)) == 507

assert np.isfinite(X).all()


print(
    "IDRiD embeddings:",
    X.shape
)


# ============================================================
# LOAD SAVED APTOS HEADS
# ============================================================

checkpoint = torch.load(
    HEADS_PATH,
    map_location="cpu",
    weights_only=False
)

assert "heads" in checkpoint

saved_heads = checkpoint["heads"]

print(
    "Available APTOS heads:",
    list(saved_heads.keys())
)


# ============================================================
# SEND EMBEDDINGS TO DEVICE
# ============================================================

X_tensor = torch.tensor(
    X,
    dtype=torch.float32,
    device=device
)


# ============================================================
# ROBUST FORWARD FUNCTION
#
# Locked architecture:
# 384 -> 512 -> ReLU -> Dropout(0.1) -> 1
#
# Dropout is OFF during inference.
#
# We identify layers by tensor shape, so this works even if
# the saved state_dict uses names such as:
# fc1.weight / net.0.weight / classifier.0.weight etc.
# ============================================================

def get_mlp_parameters(state_dict):

    first_weight = None
    first_bias = None

    final_weight = None
    final_bias = None


    # --------------------------------------------------------
    # FIND WEIGHT MATRICES
    # --------------------------------------------------------

    for key, tensor in state_dict.items():

        shape = tuple(
            tensor.shape
        )

        # 384 -> 512
        if shape == (512, 384):

            first_weight = tensor

        # 512 -> 1
        elif shape == (1, 512):

            final_weight = tensor


    # --------------------------------------------------------
    # FIND MATCHING BIASES
    # --------------------------------------------------------

    for key, tensor in state_dict.items():

        shape = tuple(
            tensor.shape
        )

        if shape == (512,):

            first_bias = tensor

        elif shape == (1,):

            final_bias = tensor


    # --------------------------------------------------------
    # CHECK
    # --------------------------------------------------------

    if first_weight is None:
        raise RuntimeError(
            "Could not find 512x384 first-layer weight."
        )

    if first_bias is None:
        raise RuntimeError(
            "Could not find 512 first-layer bias."
        )

    if final_weight is None:
        raise RuntimeError(
            "Could not find 1x512 output-layer weight."
        )

    if final_bias is None:
        raise RuntimeError(
            "Could not find output-layer bias."
        )


    return (
        first_weight,
        first_bias,
        final_weight,
        final_bias
    )


# ============================================================
# PREDICT WITH EACH FROZEN APTOS HEAD
# ============================================================

all_rows = []

print(
    "\n========== IDRiD FROZEN PREDICTIONS ==========\n"
)


with torch.no_grad():

    for seed in SEEDS:

        # ----------------------------------------------------
        # Handle integer OR string seed keys
        # ----------------------------------------------------

        if seed in saved_heads:

            head_info = saved_heads[seed]

        elif str(seed) in saved_heads:

            head_info = saved_heads[str(seed)]

        else:

            raise RuntimeError(
                f"Head for seed {seed} not found."
            )


        # ----------------------------------------------------
        # GET STATE DICT
        # ----------------------------------------------------

        if (
            isinstance(head_info, dict)
            and "state_dict" in head_info
        ):

            state_dict = head_info[
                "state_dict"
            ]

        else:

            raise RuntimeError(
                f"No state_dict found for seed {seed}"
            )


        # ----------------------------------------------------
        # EXTRACT SAVED MLP PARAMETERS
        # ----------------------------------------------------

        (
            W1,
            b1,
            W2,
            b2
        ) = get_mlp_parameters(
            state_dict
        )


        W1 = W1.to(
            device=device,
            dtype=torch.float32
        )

        b1 = b1.to(
            device=device,
            dtype=torch.float32
        )

        W2 = W2.to(
            device=device,
            dtype=torch.float32
        )

        b2 = b2.to(
            device=device,
            dtype=torch.float32
        )


        # ----------------------------------------------------
        # FORWARD PASS
        #
        # Dropout is intentionally absent because inference
        # mode disables dropout.
        # ----------------------------------------------------

        hidden = F.linear(
            X_tensor,
            W1,
            b1
        )

        hidden = F.relu(
            hidden
        )

        logits = F.linear(
            hidden,
            W2,
            b2
        ).squeeze(1)


        probabilities = torch.sigmoid(
            logits
        )


        logits_np = (
            logits
            .cpu()
            .numpy()
        )

        probabilities_np = (
            probabilities
            .cpu()
            .numpy()
        )


        # ----------------------------------------------------
        # CHECK
        # ----------------------------------------------------

        assert len(logits_np) == 507

        assert np.isfinite(
            logits_np
        ).all()

        assert np.isfinite(
            probabilities_np
        ).all()


        print(
            f"Seed {seed}: "
            f"507 predictions completed"
        )


        # ----------------------------------------------------
        # STORE
        # ----------------------------------------------------

        seed_df = pd.DataFrame({

            "seed":
                seed,

            "image_id":
                image_ids,

            "original_image_id":
                original_image_ids,

            "source_split":
                source_split,

            "original_grade":
                original_grade,

            "binary_label":
                binary_label,

            "raw_logit":
                logits_np,

            "raw_probability":
                probabilities_np
        })


        all_rows.append(
            seed_df
        )


# ============================================================
# COMBINE
# ============================================================

predictions = pd.concat(
    all_rows,
    ignore_index=True
)


# ============================================================
# FINAL CHECKS
# ============================================================

assert len(predictions) == (
    507 * 5
)

assert predictions[
    "seed"
].nunique() == 5


for seed in SEEDS:

    seed_df = predictions[
        predictions["seed"] == seed
    ]

    assert len(seed_df) == 507

    assert seed_df[
        "image_id"
    ].nunique() == 507


assert predictions[
    "raw_logit"
].notna().all()

assert predictions[
    "raw_probability"
].between(
    0,
    1
).all()


# ============================================================
# SAVE
# ============================================================

predictions.to_csv(
    OUTPUT_PATH,
    index=False
)


# ============================================================
# SUMMARY
# ============================================================

print(
    "\n========== COMPLETE ==========\n"
)

print(
    "Images per seed : 507"
)

print(
    "Number of seeds : 5"
)

print(
    "Total rows      :",
    len(predictions)
)

print(
    "NaN logits      :",
    predictions[
        "raw_logit"
    ].isna().sum()
)

print(
    "NaN probabilities:",
    predictions[
        "raw_probability"
    ].isna().sum()
)


print(
    "\n✅ Saved permanently:"
)

print(
    OUTPUT_PATH
)

print(
    "\nNo IDRiD model training performed."
)

print(
    "No IDRiD calibration performed."
)

print(
    "No IDRiD threshold selection performed."
)

Device: cuda
IDRiD embeddings: (507, 384)
Available APTOS heads: [42, 43, 44, 45, 46]

========== IDRiD FROZEN PREDICTIONS ==========

Seed 42: 507 predictions completed
Seed 43: 507 predictions completed
Seed 44: 507 predictions completed
Seed 45: 507 predictions completed
Seed 46: 507 predictions completed

========== COMPLETE ==========

Images per seed : 507
Number of seeds : 5
Total rows      : 2535
NaN logits      : 0
NaN probabilities: 0

✅ Saved permanently:
/content/drive/MyDrive/thesis_DR/idrid_retfound_raw_predictions.csv

No IDRiD model training performed.
No IDRiD calibration performed.
No IDRiD threshold selection performed.


In [23]:
# ============================================================
# IDRiD SOURCE-ONLY EXTERNAL EVALUATION - FIXED
# Uses frozen APTOS temperatures + frozen APTOS LTT thresholds
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

from scipy.special import expit
from scipy.stats import beta

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    log_loss,
    brier_score_loss
)


# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/thesis_DR"
)

PRED_PATH = (
    PROJECT_DIR
    / "idrid_retfound_raw_predictions.csv"
)

OUTPUT_PATH = (
    PROJECT_DIR
    / "idrid_external_results.csv"
)

SEEDS = [42, 43, 44, 45, 46]


# ============================================================
# FROZEN APTOS TEMPERATURES
# ============================================================

APTOS_TEMPERATURE = {
    42: 1.144044,
    43: 1.002538,
    44: 1.273578,
    45: 1.067634,
    46: 1.079971
}


# ============================================================
# FROZEN APTOS 5% LTT THRESHOLDS
# ============================================================

APTOS_LTT_THRESHOLD = {
    42: 0.159284,
    43: 0.027966,
    44: 0.046259,
    45: 0.074672,
    46: 0.097161
}


# ============================================================
# ONE-SIDED 95% CLOPPER-PEARSON UPPER BOUND
# ============================================================

def cp_upper(errors, n, confidence=0.95):

    if n == 0:
        return np.nan

    if errors == n:
        return 1.0

    return float(
        beta.ppf(
            confidence,
            errors + 1,
            n - errors
        )
    )


# ============================================================
# LOAD PREDICTIONS
# ============================================================

pred = pd.read_csv(
    PRED_PATH,
    dtype={"image_id": str}
)

assert len(pred) == 507 * 5
assert pred["seed"].nunique() == 5


print(
    "========== IDRiD SOURCE-ONLY EXTERNAL EVALUATION ==========\n"
)

print("Images per seed      : 507")
print("IDRiD training       : NO")
print("IDRiD calibration    : NO")
print("IDRiD threshold tune : NO\n")


# ============================================================
# EVALUATE
# ============================================================

rows = []


for seed in SEEDS:

    df = pred[
        pred["seed"] == seed
    ].copy()

    assert len(df) == 507


    # --------------------------------------------------------
    # LABELS + LOGITS
    # --------------------------------------------------------

    y = df[
        "binary_label"
    ].to_numpy(dtype=int)

    logits = df[
        "raw_logit"
    ].to_numpy(dtype=float)


    # --------------------------------------------------------
    # RAW PROBABILITY
    # --------------------------------------------------------

    raw_prob = expit(
        logits
    )


    # --------------------------------------------------------
    # APPLY FROZEN APTOS TEMPERATURE
    # --------------------------------------------------------

    T = APTOS_TEMPERATURE[
        seed
    ]

    calibrated_prob = expit(
        logits / T
    )


    # --------------------------------------------------------
    # APPLY FROZEN APTOS LTT THRESHOLD
    # --------------------------------------------------------

    threshold = APTOS_LTT_THRESHOLD[
        seed
    ]

    auto_clear = (
        calibrated_prob <= threshold
    )


    # ========================================================
    # SELECTIVE-RISK METRICS
    # ========================================================

    n = len(y)

    n_clear = int(
        auto_clear.sum()
    )

    errors = int(
        (
            auto_clear
            &
            (y == 1)
        ).sum()
    )

    coverage = (
        n_clear / n
    )

    review_burden = (
        1.0 - coverage
    )

    if n_clear > 0:

        auto_clear_risk = (
            errors / n_clear
        )

        auto_clear_npv = (
            1.0 - auto_clear_risk
        )

        upper_95 = cp_upper(
            errors,
            n_clear,
            confidence=0.95
        )

    else:

        auto_clear_risk = np.nan
        auto_clear_npv = np.nan
        upper_95 = np.nan


    total_referable = int(
        (y == 1).sum()
    )

    referable_miss_proportion = (
        errors / total_referable
        if total_referable > 0
        else np.nan
    )


    satisfies_5pct = bool(
        n_clear > 0
        and
        upper_95 <= 0.05
    )


    # ========================================================
    # STANDARD CLASSIFICATION METRICS
    # threshold = 0.5
    # ========================================================

    pred_class = (
        calibrated_prob >= 0.5
    ).astype(int)


    tn, fp, fn, tp = confusion_matrix(
        y,
        pred_class,
        labels=[0, 1]
    ).ravel()


    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    ppv = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else np.nan
    )

    classification_npv = (
        tn / (tn + fn)
        if (tn + fn) > 0
        else np.nan
    )


    auroc = roc_auc_score(
        y,
        calibrated_prob
    )

    auprc = average_precision_score(
        y,
        calibrated_prob
    )

    f1 = f1_score(
        y,
        pred_class,
        zero_division=0
    )

    balanced_acc = balanced_accuracy_score(
        y,
        pred_class
    )


    # ========================================================
    # CALIBRATION METRICS
    # ========================================================

    raw_nll = log_loss(
        y,
        raw_prob,
        labels=[0, 1]
    )

    calibrated_nll = log_loss(
        y,
        calibrated_prob,
        labels=[0, 1]
    )

    raw_brier = brier_score_loss(
        y,
        raw_prob
    )

    calibrated_brier = brier_score_loss(
        y,
        calibrated_prob
    )


    # ========================================================
    # STORE
    # ========================================================

    rows.append({

        "seed":
            seed,

        "temperature":
            T,

        "ltt_threshold":
            threshold,

        "n":
            n,

        "non_referable":
            int((y == 0).sum()),

        "referable":
            total_referable,

        "auto_clear":
            n_clear,

        "auto_clear_errors":
            errors,

        "coverage":
            coverage,

        "review_burden":
            review_burden,

        "auto_clear_risk":
            auto_clear_risk,

        "auto_clear_npv":
            auto_clear_npv,

        "referable_miss_proportion":
            referable_miss_proportion,

        "upper_95_risk_bound":
            upper_95,

        "satisfies_5pct":
            satisfies_5pct,

        "auroc":
            auroc,

        "auprc":
            auprc,

        "sensitivity":
            sensitivity,

        "specificity":
            specificity,

        "ppv":
            ppv,

        "classification_npv":
            classification_npv,

        "f1":
            f1,

        "balanced_accuracy":
            balanced_acc,

        "raw_nll":
            raw_nll,

        "calibrated_nll":
            calibrated_nll,

        "raw_brier":
            raw_brier,

        "calibrated_brier":
            calibrated_brier
    })


    # ========================================================
    # PRINT SEED
    # ========================================================

    print(
        f"---------- SEED {seed} ----------"
    )

    print(
        f"Temperature       : {T:.6f}"
    )

    print(
        f"LTT threshold     : {threshold:.6f}"
    )

    print(
        f"Auto-clear        : {n_clear} / 507"
    )

    print(
        f"Errors            : {errors}"
    )

    print(
        f"Coverage          : {coverage:.4f}"
    )

    print(
        f"Auto-clear risk   : {auto_clear_risk:.4f}"
    )

    print(
        f"95% upper bound   : {upper_95:.4f}"
    )

    print(
        "5% policy         :",
        "PASS" if satisfies_5pct else "FAIL"
    )

    print(
        f"AUROC             : {auroc:.4f}"
    )

    print(
        f"AUPRC             : {auprc:.4f}"
    )

    print()


# ============================================================
# RESULTS DATAFRAME
# ============================================================

results = pd.DataFrame(
    rows
)


# ============================================================
# SUMMARY
# ============================================================

print(
    "========== IDRiD SUMMARY ==========\n"
)

print(
    "5% policy satisfied :",
    f"{int(results['satisfies_5pct'].sum())} / 5"
)

print(
    f"Mean coverage        : "
    f"{results['coverage'].mean():.4f}"
)

print(
    f"Mean review burden   : "
    f"{results['review_burden'].mean():.4f}"
)

print(
    f"Mean auto-clear risk : "
    f"{results['auto_clear_risk'].mean():.4f}"
)

print(
    f"Mean 95% upper bound : "
    f"{results['upper_95_risk_bound'].mean():.4f}"
)

print(
    f"Mean AUROC           : "
    f"{results['auroc'].mean():.4f}"
)

print(
    f"Mean AUPRC           : "
    f"{results['auprc'].mean():.4f}"
)

print(
    f"Mean sensitivity     : "
    f"{results['sensitivity'].mean():.4f}"
)

print(
    f"Mean specificity     : "
    f"{results['specificity'].mean():.4f}"
)

print(
    f"Mean PPV             : "
    f"{results['ppv'].mean():.4f}"
)

print(
    f"Mean NPV             : "
    f"{results['classification_npv'].mean():.4f}"
)

print(
    f"Mean F1              : "
    f"{results['f1'].mean():.4f}"
)

print(
    f"Mean balanced acc.   : "
    f"{results['balanced_accuracy'].mean():.4f}"
)


print(
    "\nCalibration:"
)

print(
    f"Mean NLL   : "
    f"{results['raw_nll'].mean():.4f}"
    f" -> "
    f"{results['calibrated_nll'].mean():.4f}"
)

print(
    f"Mean Brier : "
    f"{results['raw_brier'].mean():.4f}"
    f" -> "
    f"{results['calibrated_brier'].mean():.4f}"
)


# ============================================================
# SAVE
# ============================================================

results.to_csv(
    OUTPUT_PATH,
    index=False
)


print(
    "\n✅ Saved permanently:"
)

print(
    OUTPUT_PATH
)

print(
    "\n========== COMPLETE =========="
)

========== IDRiD SOURCE-ONLY EXTERNAL EVALUATION ==========

Images per seed      : 507
IDRiD training       : NO
IDRiD calibration    : NO
IDRiD threshold tune : NO

---------- SEED 42 ----------
Temperature       : 1.144044
LTT threshold     : 0.159284
Auto-clear        : 173 / 507
Errors            : 28
Coverage          : 0.3412
Auto-clear risk   : 0.1618
95% upper bound   : 0.2151
5% policy         : FAIL
AUROC             : 0.9376
AUPRC             : 0.9657

---------- SEED 43 ----------
Temperature       : 1.002538
LTT threshold     : 0.027966
Auto-clear        : 184 / 507
Errors            : 28
Coverage          : 0.3629
Auto-clear risk   : 0.1522
95% upper bound   : 0.2026
5% policy         : FAIL
AUROC             : 0.9466
AUPRC             : 0.9698

---------- SEED 44 ----------
Temperature       : 1.273578
LTT threshold     : 0.046259
Auto-clear        : 142 / 507
Errors            : 17
Coverage          : 0.2801
Auto-clear risk   : 0.1197
95% upper bound   : 0.1742
5% poli